In [1]:
import pandas as pd
import numpy as np

garch_forecasts = pd.read_csv("../data/garch_forecasts.csv", index_col=0, parse_dates=True)

print(f"Total rows: {len(garch_forecasts)}")
print(f"Date range: {garch_forecasts.index.min()} to {garch_forecasts.index.max()}")
print(f"\nSPX forecast dates (non-null): {garch_forecasts['spx'].dropna().index.min()} to {garch_forecasts['spx'].dropna().index.max()}, count={garch_forecasts['spx'].dropna().shape[0]}")

Total rows: 2669
Date range: 2017-09-11 00:00:00 to 2024-12-31 00:00:00

SPX forecast dates (non-null): 2018-01-04 00:00:00 to 2024-12-31 00:00:00, count=1759


In [2]:
def generate_walkforward_splits(n_obs, initial_train=252, refit_every=21):
    splits = []
    
    for i in range(initial_train, n_obs, refit_every):
        train_end_idx = i  # train on indices [0, i)
        test_start_idx = i
        test_end_idx = min(i + refit_every, n_obs)  # exclusive
        
        splits.append((train_end_idx, test_start_idx, test_end_idx))
    
    return splits

# Test it on SPX's length
spx_returns_len = 2011  # from the GARCH notebook's Cell 1 output
splits = generate_walkforward_splits(spx_returns_len)

print(f"Generated {len(splits)} splits")
print(f"First split: {splits[0]}")
print(f"Last split: {splits[-1]}")

Generated 84 splits
First split: (252, 252, 273)
Last split: (1995, 1995, 2011)


In [3]:
# Convert split index positions into actual dates using SPX's returns index
spx = pd.read_csv("../data/spx.csv", index_col=0, parse_dates=True)
spx_returns_index = spx.index[1:]  # matches GARCH notebook: first row dropped for return calc

# Get the first test date from each split
generated_test_dates = [spx_returns_index[test_start] for (_, test_start, _) in splits]

# Compare to GARCH's actual first-of-each-block forecast dates
garch_spx_dates = garch_forecasts["spx"].dropna().index

# Check: does every generated test start date actually appear in GARCH's forecast dates?
matches = [d in garch_spx_dates for d in generated_test_dates]
print(f"Generated dates matching GARCH forecast dates: {sum(matches)} / {len(matches)}")

if sum(matches) != len(matches):
    print("Mismatched dates:", [d for d, m in zip(generated_test_dates, matches) if not m])

Generated dates matching GARCH forecast dates: 84 / 84


In [4]:
def get_train_test_dates(dates_index, split):
    train_end_idx, test_start_idx, test_end_idx = split
    
    train_dates = (dates_index[0], dates_index[train_end_idx - 1])  # (start, end) of training window
    test_dates = (dates_index[test_start_idx], dates_index[test_end_idx - 1])  # (start, end) of test window
    
    return train_dates, test_dates

# Try it on the first split
train_dates, test_dates = get_train_test_dates(spx_returns_index, splits[0])
print(f"Train: {train_dates[0]} to {train_dates[1]}")
print(f"Test: {test_dates[0]} to {test_dates[1]}")

Train: 2017-01-04 00:00:00 to 2018-01-03 00:00:00
Test: 2018-01-04 00:00:00 to 2018-02-02 00:00:00


In [5]:
print("First 3 splits:")
for split in splits[:3]:
    train_dates, test_dates = get_train_test_dates(spx_returns_index, split)
    print(f"  Train: {train_dates[0].date()} to {train_dates[1].date()} | Test: {test_dates[0].date()} to {test_dates[1].date()}")

print("\nLast 3 splits:")
for split in splits[-3:]:
    train_dates, test_dates = get_train_test_dates(spx_returns_index, split)
    print(f"  Train: {train_dates[0].date()} to {train_dates[1].date()} | Test: {test_dates[0].date()} to {test_dates[1].date()}")

First 3 splits:
  Train: 2017-01-04 to 2018-01-03 | Test: 2018-01-04 to 2018-02-02
  Train: 2017-01-04 to 2018-02-02 | Test: 2018-02-05 to 2018-03-06
  Train: 2017-01-04 to 2018-03-06 | Test: 2018-03-07 to 2018-04-05

Last 3 splits:
  Train: 2017-01-04 to 2024-10-08 | Test: 2024-10-09 to 2024-11-06
  Train: 2017-01-04 to 2024-11-06 | Test: 2024-11-07 to 2024-12-06
  Train: 2017-01-04 to 2024-12-06 | Test: 2024-12-09 to 2024-12-31


In [6]:
split_records = []
for split in splits:
    train_dates, test_dates = get_train_test_dates(spx_returns_index, split)
    split_records.append({
        "train_start": train_dates[0],
        "train_end": train_dates[1],
        "test_start": test_dates[0],
        "test_end": test_dates[1]
    })

cv_splits_df = pd.DataFrame(split_records)
cv_splits_df.to_csv("../data/cv_splits.csv", index=False)

print(f"Saved {len(cv_splits_df)} splits to cv_splits.csv")
cv_splits_df.head()

Saved 84 splits to cv_splits.csv


,train_start,train_end,test_start,test_end
0,2017-01-04,2018-01-03,2018-01-04,2018-02-02
1,2017-01-04,2018-02-02,2018-02-05,2018-03-06
2,2017-01-04,2018-03-06,2018-03-07,2018-04-05
3,2017-01-04,2018-04-05,2018-04-06,2018-05-04
4,2017-01-04,2018-05-04,2018-05-07,2018-06-05
